# Recognizing microwave activity from household power

A household electricity meter measures several appliances together. This notebook asks whether that combined signal can reveal **when the microwave is ON**, including periods when other appliances draw substantial power.

**UK-DALE** is a public dataset of electricity demand from UK homes. It contains whole-house measurements and separate appliance channels. Here, real power P and apparent power S from the household meter provide the model inputs; the dedicated microwave channel provides the training and evaluation labels. The model does not receive that appliance channel as an input. This is a device-state recognition task within non-intrusive load monitoring (NILM).

Two temporal convolutional network (**TCN**) workflows are implemented:

| Workflow | Input resolution | Output step | Recorded runs |
|---|---|---|---|
| 10-second power inputs, TCN/PQ (V6) | 10 s | 10 s | Five seeds |
| Native 1-second inputs, M0_TCN-F0 (V11 `stage2_cold2`) | 1 s | 10 s | Five seeds |

Both use a 640-second history. Each prediction is an ON-state score for an eligible endpoint; a threshold fitted on the calibration partition converts the score to an ON/OFF decision. A causal window uses current and past observations, without using future measurements. All five seeds are 11, 29, 42, 73 and 101.

The sections follow the complete route: **read mains and appliance records → build labels and electrical features → fit training-only normalization → define the TCN → train and calibrate → evaluate fixed-threshold predictions**. Model definitions and training functions appear in the notebook; the linked workflows prepare full raw-data runs.

All `RUN_*` flags initially equal `False`. Default execution defines the workflow and displays recorded results. Data reading, checkpoint loading and training require their corresponding flags; training also requires local data and a new output directory.

[Methods](../docs/en/methods.md) · [Data and dataset sources](../docs/en/data.md)


In [ ]:
from __future__ import annotations
import copy, hashlib, json, math, random, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_fscore_support, balanced_accuracy_score, confusion_matrix
sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'results' / 'index.json').is_file() and (p / 'src' / 'wattsign').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the WattSign repository')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from wattsign.notebook_view import show_experiment
try:
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import Dataset, DataLoader, Subset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
RUN_DATA_READ = False
RUN_MODEL_LOAD = False
RUN_TRAINING = False
DATASET_ROOT = ''
ARTIFACT_ROOT = ''
NEW_OUTPUT_ROOT = ''
MICROWAVE_CHANNEL_FILE = ''
TRAINING_DEVICE = 'cuda'
EVALUATION_SPECS = []
CASE_TO_TRAIN = 'ukdale-1s'
CHECKPOINT_SEED = 42
SEEDS = (11, 29, 42, 73, 101)
RULE = {'on_threshold_w': 800.0, 'off_threshold_w': 100.0, 'minimum_activity_seconds': 20.0, 'short_off_merge_seconds': 15.0, 'maximum_sample_gap_seconds': 30.0}
MICROWAVE_RULE = RULE
print('Execution flags: data reading, model loading and training disabled.')

## 1. Read mains power and derive Q

Real power P describes average energy transfer; apparent power S combines voltage and current magnitudes. Their relationship adds information beyond the household's total power level, which can help distinguish loads drawing similar real power.

The reader uses the timestamp, real power **P** (W) and apparent power **S** (VA) columns in UK-DALE `mains.dat`. Before aggregation it derives a non-active-power magnitude/proxy **Q = sqrt(max(S²−P², 0))** from these scalar measurements. This proxy does not measure signed reactive power and can include distortion-related non-active components. Invalid P/S combinations remain masked. Nonmonotonic timestamps and gaps longer than 30 s start new continuous segments. V6 and V11 use separate aggregation implementations.

The optional read cell accepts a local dataset path and a bounded time interval.


In [ ]:
def scan_psq_10s(path: Path, start_s: int, end_s: int) -> dict:
    n1, n10 = (end_s - start_s, (end_s - start_s) // 10)
    p = np.zeros(n10, np.float32)
    s = np.zeros(n10, np.float32)
    q = np.zeros(n10, np.float32)
    pv = np.zeros(n10, bool)
    sv = np.zeros(n10, bool)
    qv = np.zeros(n10, bool)
    p1_valid = np.zeros(n1, bool)
    break1 = np.zeros(n1, bool)
    break10 = np.zeros(n10, bool)
    audit = {'raw_records': 0, 'records_in_interval': 0, 'nonmonotonic_records': 0, 'recovery_breaks': 0, 'gaps_over_30s': 0, 'invalid_raw_q_records': 0}
    carry = pd.DataFrame()
    previous_kept = None
    recovery_pending = False
    reached_end = False

    def consume(frame: pd.DataFrame) -> None:
        if frame.empty:
            return
        grouped = frame.groupby('bin10', sort=False).agg(p=('p_clean', 'median'), p_bad=('p_bad', 'max'), s=('s_clean', 'median'), q=('q_clean', 'median'))
        idx = grouped.index.to_numpy(np.int64)
        ok = (idx >= 0) & (idx < n10)
        idx = idx[ok]
        grouped = grouped.iloc[np.flatnonzero(ok)]
        pp = grouped['p'].to_numpy(float)
        ss = grouped['s'].to_numpy(float)
        qq = grouped['q'].to_numpy(float)
        pgood = np.isfinite(pp) & ~grouped['p_bad'].to_numpy(bool)
        sgood = np.isfinite(ss)
        qgood = np.isfinite(qq)
        p[idx[pgood]] = pp[pgood].astype(np.float32)
        pv[idx[pgood]] = True
        s[idx[sgood]] = ss[sgood].astype(np.float32)
        sv[idx[sgood]] = True
        q[idx[qgood]] = qq[qgood].astype(np.float32)
        qv[idx[qgood]] = True
        grouped1 = frame.groupby('bin1', sort=False).agg(p=('p_clean', 'median'), p_bad=('p_bad', 'max'))
        idx1 = grouped1.index.to_numpy(np.int64)
        ok1 = (idx1 >= 0) & (idx1 < n1)
        idx1 = idx1[ok1]
        grouped1 = grouped1.iloc[np.flatnonzero(ok1)]
        p1good = np.isfinite(grouped1['p'].to_numpy(float)) & ~grouped1['p_bad'].to_numpy(bool)
        p1_valid[idx1[p1good]] = True
    for chunk in pd.read_csv(path, sep='\\s+', header=None, usecols=[0, 1, 2], names=['ts', 'p', 's'], chunksize=500000):
        ts_raw = pd.to_numeric(chunk['ts'], errors='coerce').to_numpy(float)
        pp_raw = pd.to_numeric(chunk['p'], errors='coerce').to_numpy(float)
        ss_raw = pd.to_numeric(chunk['s'], errors='coerce').to_numpy(float)
        finite_ts = np.isfinite(ts_raw)
        ts_raw, pp_raw, ss_raw = (ts_raw[finite_ts], pp_raw[finite_ts], ss_raw[finite_ts])
        audit['raw_records'] += int(ts_raw.size)
        if ts_raw.size == 0:
            continue
        prior = np.maximum.accumulate(np.r_[-np.inf if previous_kept is None else previous_kept, ts_raw[:-1]])
        accepted_mask = ts_raw > prior
        accepted_index = np.flatnonzero(accepted_mask)
        audit['nonmonotonic_records'] += int((~accepted_mask).sum())
        if accepted_index.size == 0:
            recovery_pending = True
            continue
        recovery = np.zeros(accepted_index.size, bool)
        if recovery_pending or (~accepted_mask).any():
            pending = recovery_pending
            previous_raw_index = -1
            for accepted_position, raw_index in enumerate(accepted_index):
                if raw_index > previous_raw_index + 1:
                    pending = True
                if pending and start_s <= ts_raw[raw_index] < end_s:
                    recovery[accepted_position] = True
                    pending = False
                previous_raw_index = int(raw_index)
            if accepted_index[-1] < ts_raw.size - 1:
                pending = True
            recovery_pending = pending
        else:
            recovery_pending = False
        ts, pp, ss = (ts_raw[accepted_index], pp_raw[accepted_index], ss_raw[accepted_index])
        diffs = np.diff(np.r_[previous_kept if previous_kept is not None else ts[0], ts])
        gaps = diffs > MICROWAVE_RULE['maximum_sample_gap_seconds']
        in_span = (ts >= start_s) & (ts < end_s)
        boundary = (gaps | recovery) & in_span
        if boundary.any():
            boundary_ts = ts[boundary]
            i1 = np.floor(boundary_ts).astype(np.int64) - start_s
            i10 = (np.floor(boundary_ts / 10).astype(np.int64) * 10 - start_s) // 10
            break1[i1] = True
            break10[i10] = True
        audit['gaps_over_30s'] += int((gaps & in_span).sum())
        audit['recovery_breaks'] += int((recovery & in_span).sum())
        previous_kept = float(ts[-1])
        reached_end = bool((ts >= end_s).any())
        ts, pp, ss = (ts[in_span], pp[in_span], ss[in_span])
        audit['records_in_interval'] += int(ts.size)
        if ts.size:
            p_bad = ~np.isfinite(pp) | np.isfinite(pp) & np.isfinite(ss) & (pp == 0.0) & (ss > 50.0)
            s_bad = ~np.isfinite(ss) | (ss < 0.0)
            q_ok = ~p_bad & ~s_bad & (ss >= np.abs(pp))
            q_raw = np.where(q_ok, np.sqrt(np.maximum(ss * ss - pp * pp, 0.0)), np.nan)
            audit['invalid_raw_q_records'] += int((~q_ok).sum())
            frame = pd.DataFrame({'bin10': (np.floor(ts / 10).astype(np.int64) * 10 - start_s) // 10, 'bin1': np.floor(ts).astype(np.int64) - start_s, 'p_clean': np.where(p_bad, np.nan, pp), 'p_bad': p_bad, 's_clean': np.where(s_bad, np.nan, ss), 'q_clean': q_raw})
            if not carry.empty:
                frame = pd.concat([carry, frame], ignore_index=True)
            last_bin = int(frame['bin10'].iloc[-1])
            complete = frame['bin10'].to_numpy() != last_bin
            consume(frame.loc[complete])
            carry = frame.loc[~complete].copy()
        if reached_end:
            break
    consume(carry)
    break1[0] = False
    break10[0] = False
    continuity1 = np.cumsum(break1, dtype=np.int32)
    continuity10 = np.cumsum(break10, dtype=np.int32)
    audit.update({'valid_p_bins_1s': int(p1_valid.sum()), 'valid_p_bins_10s': int(pv.sum()), 'valid_s_bins_10s': int(sv.sum()), 'valid_q_bins_10s': int(qv.sum()), 'segments_1s': int(continuity1.max() + 1), 'segments_10s': int(continuity10.max() + 1), 'q_coverage_fraction': float(qv.mean())})
    return {'p': p, 's': s, 'q': q, 'p_valid': pv, 's_valid': sv, 'q_valid': qv, 'p1_valid': p1_valid, 'break1': break1, 'break10': break10, 'continuity1': continuity1, 'continuity10': continuity10, 'breaks': break10, 'audit': audit}

In [ ]:
def scan_psq_seconds(path, start_s, end_s, gains):
    """Frozen H2 P/Q reader: transform individual raw records before aggregation."""
    n1, n10 = (end_s - start_s, (end_s - start_s) // 10)
    p1 = np.zeros(n1, np.float32)
    p10 = np.zeros(n10, np.float32)
    p1_valid = np.zeros(n1, bool)
    p10_valid = np.zeros(n10, bool)
    p1_bad = np.zeros(n1, bool)
    p10_bad = np.zeros(n10, bool)
    q1 = {g: np.zeros(n1, np.float32) for g in gains}
    q10 = {g: np.zeros(n10, np.float32) for g in gains}
    q1_valid = {g: np.zeros(n1, bool) for g in gains}
    q10_valid = {g: np.zeros(n10, bool) for g in gains}
    break1 = np.zeros(n1, bool)
    break10 = np.zeros(n10, bool)
    current1 = current10 = None
    p_values1, p_values10 = ([], [])
    q_values1, q_values10 = ({g: [] for g in gains}, {g: [] for g in gains})
    previous, recovery = (None, False)
    audit = {'accepted_records': 0, 'nonmonotonic': 0, 'gap_over_30': 0, 'first_excluded_timestamp': None}

    def flush1():
        nonlocal current1, p_values1, q_values1
        if current1 is not None:
            if p_values1 and (not p1_bad[current1]):
                p1[current1] = np.float32(np.median(np.asarray(p_values1, np.float64)))
                p1_valid[current1] = True
            for gain in gains:
                if q_values1[gain]:
                    q1[gain][current1] = np.float32(np.median(np.asarray(q_values1[gain], np.float64)))
                    q1_valid[gain][current1] = True
        current1 = None
        p_values1 = []
        q_values1 = {g: [] for g in gains}

    def flush10():
        nonlocal current10, p_values10, q_values10
        if current10 is not None:
            if p_values10 and (not p10_bad[current10]):
                p10[current10] = np.float32(np.median(np.asarray(p_values10, np.float64)))
                p10_valid[current10] = True
            for gain in gains:
                if q_values10[gain]:
                    q10[gain][current10] = np.float32(np.median(np.asarray(q_values10[gain], np.float64)))
                    q10_valid[gain][current10] = True
        current10 = None
        p_values10 = []
        q_values10 = {g: [] for g in gains}
    with Path(path).open('r', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            fields = line.split()
            if len(fields) < 3:
                continue
            try:
                timestamp = float(fields[0])
            except ValueError:
                continue
            if not math.isfinite(timestamp):
                continue
            if timestamp >= end_s:
                audit['first_excluded_timestamp'] = timestamp
                break
            try:
                p, s = (float(fields[1]), float(fields[2]))
            except ValueError:
                continue
            if previous is not None and timestamp <= previous:
                audit['nonmonotonic'] += 1
                recovery = True
                continue
            gap = previous is not None and timestamp - previous > RULE['maximum_sample_gap_seconds']
            if timestamp < start_s:
                previous = timestamp
                continue
            i1 = int(math.floor(timestamp)) - start_s
            i10 = (int(math.floor(timestamp / 10)) * 10 - start_s) // 10
            if not (0 <= i1 < n1 and 0 <= i10 < n10):
                continue
            if gap or recovery:
                break1[i1] = True
                break10[i10] = True
                audit['gap_over_30'] += int(gap)
                recovery = False
            if current1 is None:
                current1 = i1
            elif current1 != i1:
                flush1()
                current1 = i1
            if current10 is None:
                current10 = i10
            elif current10 != i10:
                flush10()
                current10 = i10
            p_bad = not math.isfinite(p) or (math.isfinite(s) and p == 0.0 and (s > 50.0))
            if p_bad:
                p1_bad[i1] = True
                p10_bad[i10] = True
            else:
                p_values1.append(p)
                p_values10.append(p)
            for gain in gains:
                scaled_s = gain * s
                if not p_bad and math.isfinite(scaled_s) and (scaled_s >= abs(p)):
                    q = math.sqrt(max(scaled_s * scaled_s - p * p, 0.0))
                    q_values1[gain].append(q)
                    q_values10[gain].append(q)
            audit['accepted_records'] += 1
            previous = timestamp
    flush1()
    flush10()
    break1[0] = False
    break10[0] = False
    assert audit['first_excluded_timestamp'] is not None and audit['first_excluded_timestamp'] >= end_s
    return {'p1': p1, 'p1_valid': p1_valid, 'p10': p10, 'p10_valid': p10_valid, 'q1': q1, 'q10': q10, 'q1_valid': q1_valid, 'q10_valid': q10_valid, 'continuity1': np.cumsum(break1, dtype=np.int32), 'continuity10': np.cumsum(break10, dtype=np.int32), 'audit': audit}

### Dedicated-channel labels

Microwave labels use 800 W ON / 100 W OFF hysteresis, a 20 s minimum activity duration, 15 s short-OFF merging and a 30 s maximum sample gap. Coverage, transition uncertainty and the ±10 s stability guard are represented separately. Labels come from the dedicated appliance channel; inputs come from aggregate mains.


In [ ]:
def _mark(array: np.ndarray, left: int, right: int, value=True) -> None:
    left, right = (max(0, int(left)), min(array.size, int(right)))
    if left < right:
        array[left:right] = value

def _ceil_index(value: float) -> int:
    return int(math.ceil(value - 1e-10))

def _floor_index(value: float) -> int:
    return int(math.floor(value + 1e-10))

def scan_microwave_native(path: Path, start_s: int, end_s: int) -> dict:
    stop_s = float(end_s + 2 * MICROWAVE_RULE['maximum_sample_gap_seconds'])
    previous_timestamp = None
    state = 'unknown'
    active_start = active_on_prev = trusted_start = None
    coverage_segments, raw_intervals = ([], [])
    recovery_break_pending = False
    audit = {'native_records': 0, 'native_gaps': 0, 'native_nonmonotonic_records': 0}

    def close_active(end_timestamp, reason, off_prev):
        nonlocal active_start, active_on_prev
        if active_start is not None and end_timestamp > active_start:
            raw_intervals.append({'start_s': float(active_start), 'end_s': float(end_timestamp), 'end_reason': reason, 'on_prev_s': float(active_on_prev if active_on_prev is not None else active_start), 'off_prev_s': float(off_prev if off_prev is not None else end_timestamp)})
        active_start = active_on_prev = None
    reached_stop = False
    for chunk in pd.read_csv(path, sep='\\s+', header=None, usecols=[0, 1], names=['ts', 'power'], chunksize=250000):
        for ts, power in zip(pd.to_numeric(chunk['ts'], errors='coerce').to_numpy(float), pd.to_numeric(chunk['power'], errors='coerce').to_numpy(float), strict=True):
            if not (math.isfinite(ts) and math.isfinite(power)):
                continue
            if ts > stop_s:
                reached_stop = True
                break
            audit['native_records'] += 1
            if previous_timestamp is not None and ts <= previous_timestamp:
                audit['native_nonmonotonic_records'] += 1
                recovery_break_pending = True
                continue
            is_gap = previous_timestamp is not None and ts - previous_timestamp > MICROWAVE_RULE['maximum_sample_gap_seconds']
            if is_gap or recovery_break_pending:
                audit['native_gaps'] += int(is_gap)
                if active_start is not None and previous_timestamp is not None:
                    close_active(previous_timestamp, 'data_gap', previous_timestamp)
                if trusted_start is not None and previous_timestamp is not None:
                    coverage_segments.append((trusted_start, previous_timestamp))
                state = 'unknown'
                trusted_start = None
                recovery_break_pending = False
            if state == 'unknown' and power <= MICROWAVE_RULE['off_threshold_w']:
                state = 'off'
                trusted_start = float(ts)
            elif state == 'off' and power >= MICROWAVE_RULE['on_threshold_w']:
                active_start = float(ts)
                active_on_prev = float(previous_timestamp if previous_timestamp is not None else ts)
                state = 'on'
            elif state == 'on' and power <= MICROWAVE_RULE['off_threshold_w']:
                close_active(ts, 'off_threshold', previous_timestamp if previous_timestamp is not None else ts)
                state = 'off'
            previous_timestamp = float(ts)
        if reached_stop:
            break
    if active_start is not None and previous_timestamp is not None:
        close_active(previous_timestamp, 'right_censored', previous_timestamp)
    if trusted_start is not None and previous_timestamp is not None:
        coverage_segments.append((trusted_start, previous_timestamp))
    raw_intervals.sort(key=lambda x: (x['start_s'], x['end_s']))
    merged = []
    for interval in raw_intervals:
        if merged and merged[-1]['end_reason'] == 'off_threshold' and (0 <= interval['start_s'] - merged[-1]['end_s'] <= MICROWAVE_RULE['short_off_merge_seconds']):
            merged[-1].update({'end_s': interval['end_s'], 'end_reason': interval['end_reason'], 'off_prev_s': interval['off_prev_s']})
        else:
            merged.append(dict(interval))
    accepted = []
    for interval in merged:
        interval['span_seconds'] = interval['end_s'] - interval['start_s']
        if interval['span_seconds'] >= MICROWAVE_RULE['minimum_activity_seconds']:
            accepted.append(interval)
    audit.update({'accepted_events': len(accepted), 'merged_intervals': len(merged), 'trusted_coverage_segments': len(coverage_segments)})
    return {'coverage_segments': coverage_segments, 'merged_intervals': merged, 'accepted_intervals': accepted, 'audit': audit}

def project_labels_to_10s(scan: dict, start_s: int, end_s: int) -> dict:
    n1 = end_s - start_s
    covered = np.zeros(n1, bool)
    overlaps = np.zeros(n1, bool)
    accepted = np.zeros(n1, bool)
    event = np.full(n1, -1, np.int32)
    uncertain = np.zeros(n1, bool)
    guard = np.zeros(n1, bool)
    for left, right in scan['coverage_segments']:
        _mark(covered, _ceil_index(left - start_s), _floor_index(right - start_s))
    for interval in scan['merged_intervals']:
        _mark(overlaps, _floor_index(interval['start_s'] - start_s), _ceil_index(interval['end_s'] - start_s))
    for eid, interval in enumerate(scan['accepted_intervals']):
        left, right = (_ceil_index(interval['start_s'] - start_s), _floor_index(interval['end_s'] - start_s))
        _mark(accepted, left, right)
        _mark(event, left, right, eid)
        _mark(uncertain, math.floor(interval['on_prev_s'] - start_s), math.ceil(interval['start_s'] - start_s))
        if interval['end_reason'] == 'off_threshold':
            _mark(uncertain, math.floor(interval['off_prev_s'] - start_s), math.ceil(interval['end_s'] - start_s))
        for boundary in (interval['start_s'], interval['end_s']):
            _mark(guard, math.floor(boundary - 10 - start_s), math.ceil(boundary + 10 - start_s))
    target = (covered & accepted).astype(np.uint8)
    valid = covered & (~overlaps | accepted)
    supervised = valid & ~uncertain
    stable = supervised & ~guard
    endpoint = np.arange(9, n1, 10)
    return {'target': target[endpoint], 'valid': supervised[endpoint].astype(np.uint8), 'stable_valid': stable[endpoint].astype(np.uint8), 'event_id': event[endpoint]}

def scan_microwave_labels(path, start_s, end_s):
    """Dedicated microwave labels with the 800/100/20/15/30 hysteresis and duration rules."""
    stop_s = end_s + 2 * RULE['maximum_sample_gap_seconds']
    previous = None
    state = 'unknown'
    active = on_previous = trusted = None
    coverage, intervals, recovery = ([], [], False)

    def close(end, reason, off_previous):
        nonlocal active, on_previous
        if active is not None and end > active:
            intervals.append({'start': active, 'end': end, 'reason': reason, 'on_previous': on_previous if on_previous is not None else active, 'off_previous': off_previous if off_previous is not None else end})
        active = on_previous = None
    with Path(path).open('r', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            fields = line.split()
            if len(fields) < 2:
                continue
            try:
                timestamp, power = (float(fields[0]), float(fields[1]))
            except ValueError:
                continue
            if not (math.isfinite(timestamp) and math.isfinite(power)):
                continue
            if timestamp > stop_s:
                break
            if previous is not None and timestamp <= previous:
                recovery = True
                continue
            gap = previous is not None and timestamp - previous > RULE['maximum_sample_gap_seconds']
            if gap or recovery:
                if active is not None and previous is not None:
                    close(previous, 'gap', previous)
                if trusted is not None and previous is not None:
                    coverage.append((trusted, previous))
                state = 'unknown'
                trusted = None
                recovery = False
            if state == 'unknown' and power <= RULE['off_threshold_w']:
                state = 'off'
                trusted = timestamp
            elif state == 'off' and power >= RULE['on_threshold_w']:
                active = timestamp
                on_previous = previous if previous is not None else timestamp
                state = 'on'
            elif state == 'on' and power <= RULE['off_threshold_w']:
                close(timestamp, 'off', previous if previous is not None else timestamp)
                state = 'off'
            previous = timestamp
    if active is not None and previous is not None:
        close(previous, 'censored', previous)
    if trusted is not None and previous is not None:
        coverage.append((trusted, previous))
    intervals.sort(key=lambda row: (row['start'], row['end']))
    merged = []
    for row in intervals:
        if merged and merged[-1]['reason'] == 'off' and (0 <= row['start'] - merged[-1]['end'] <= RULE['short_off_merge_seconds']):
            merged[-1].update(end=row['end'], reason=row['reason'], off_previous=row['off_previous'])
        else:
            merged.append(dict(row))
    accepted = [row for row in merged if row['end'] - row['start'] >= RULE['minimum_activity_seconds']]
    n1 = end_s - start_s
    covered = np.zeros(n1, bool)
    overlaps = np.zeros(n1, bool)
    active_mask = np.zeros(n1, bool)
    event = np.full(n1, -1, np.int32)
    uncertain = np.zeros(n1, bool)
    stable_guard = np.zeros(n1, bool)

    def mark(array, left, right, value=True):
        left, right = (max(0, int(left)), min(n1, int(right)))
        if left < right:
            array[left:right] = value
    for left, right in coverage:
        mark(covered, math.ceil(left - start_s - 1e-10), math.floor(right - start_s + 1e-10))
    for row in merged:
        mark(overlaps, math.floor(row['start'] - start_s + 1e-10), math.ceil(row['end'] - start_s - 1e-10))
    for event_id, row in enumerate(accepted):
        left = math.ceil(row['start'] - start_s - 1e-10)
        right = math.floor(row['end'] - start_s + 1e-10)
        mark(active_mask, left, right)
        event[max(0, left):min(n1, right)] = event_id
        mark(uncertain, math.floor(row['on_previous'] - start_s), math.ceil(row['start'] - start_s))
        if row['reason'] == 'off':
            mark(uncertain, math.floor(row['off_previous'] - start_s), math.ceil(row['end'] - start_s))
        for boundary in (row['start'], row['end']):
            mark(stable_guard, math.floor(boundary - 10 - start_s), math.ceil(boundary + 10 - start_s))
    valid = covered & (~overlaps | active_mask)
    supervised = valid & ~uncertain
    stable = supervised & ~stable_guard
    return {'target': (active_mask & covered)[9::10].astype(np.uint8), 'valid': supervised[9::10], 'stable': stable[9::10], 'event': event[9::10], 'accepted': accepted}

In [ ]:
START_UTC_S = 1363564800
END_UTC_S = START_UTC_S + 600
if RUN_DATA_READ:
    if not DATASET_ROOT.strip():
        raise ValueError('Set DATASET_ROOT explicitly')
    mains = Path(DATASET_ROOT).expanduser().resolve() / 'house_1' / 'mains.dat'
    if not mains.is_file():
        raise FileNotFoundError(f'Expected existing house_1/mains.dat{mains}')
    if not 0 < END_UTC_S - START_UTC_S <= 3600 or START_UTC_S % 10 or END_UTC_S % 10:
        raise ValueError('Use a 10-second aligned interval of at most one hour')
    raw10 = scan_psq_10s(mains, START_UTC_S, END_UTC_S)
    raw1 = scan_psq_seconds(mains, START_UTC_S, END_UTC_S, gains=(1.0,))
    print({'10s_valid_P': int(raw10['p_valid'].sum()), '1s_valid_P': int(raw1['p1_valid'].sum()), 'raw_Q_gain': 1.0})
else:
    print('Raw-data reading: disabled (RUN_DATA_READ=False).')

## 2. Causal features and validity masks

Each P/S/Q group contains `[value, first difference, value minus past median, value mask, difference mask, median mask]`, giving 18 channels. The **PQ** configuration zeroes all six S channels. Its input combines power values, temporal changes and missingness.

For 10 s inputs, the median uses the previous six bins and requires five valid bins. For 1 s inputs, the 60 s rolling median is shifted by one second, requires 50 valid samples and restarts at each continuous segment. Features use past observations without interpolation.


In [ ]:
def build_signal_features(value: np.ndarray, valid: np.ndarray, continuity: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n = value.size
    values = np.zeros((n, 3), np.float32)
    masks = np.zeros((n, 3), np.uint8)
    values[:, 0] = value
    masks[:, 0] = valid
    for i in range(1, n):
        if valid[i] and valid[i - 1] and (continuity[i] == continuity[i - 1]):
            values[i, 1] = np.float32(float(value[i]) - float(value[i - 1]))
            masks[i, 1] = 1
        history = np.arange(max(0, i - 6), i, dtype=np.int64)
        history = history[valid[history] & (continuity[history] == continuity[i])]
        if valid[i] and i >= 6 and (history.size >= 5):
            values[i, 2] = np.float32(float(value[i]) - float(np.median(value[history])))
            masks[i, 2] = 1
    return (values, masks)

def pack_18_channels(signals: list[tuple[np.ndarray, np.ndarray]], continuity: np.ndarray) -> np.ndarray:
    groups = []
    for value, valid in signals:
        values, masks = build_signal_features(value, valid, continuity)
        groups.append(np.concatenate([values, masks.astype(np.float32)], axis=1))
    result = np.concatenate(groups, axis=1).astype(np.float32)
    assert result.shape[1] == 18
    return result

In [ ]:
def causal_signal_features(value, valid, continuity, lookback=60, minimum=50):
    """Use only valid observations before the current bin; reset each continuous segment."""
    n = len(value)
    values = np.zeros((n, 3), np.float32)
    masks = np.zeros((n, 3), np.uint8)
    values[:, 0] = value
    masks[:, 0] = valid
    same = np.zeros(n, bool)
    same[1:] = valid[1:] & valid[:-1] & (continuity[1:] == continuity[:-1])
    positions = np.flatnonzero(same)
    values[positions, 1] = value[positions] - value[positions - 1]
    masks[positions, 1] = 1
    borders = np.flatnonzero(np.r_[True, continuity[1:] != continuity[:-1], True])
    for left, right in zip(borders[:-1], borders[1:], strict=True):
        series = np.where(valid[left:right], value[left:right], np.nan)
        baseline = pd.Series(series).rolling(lookback, min_periods=minimum).median().shift(1).to_numpy(copy=True)
        baseline[:min(lookback, len(baseline))] = np.nan
        good = valid[left:right] & np.isfinite(baseline)
        local = np.flatnonzero(good)
        values[left + local, 2] = value[left + local] - baseline[local]
        masks[left + local, 2] = 1
    return np.concatenate([values, masks.astype(np.float32)], axis=1)

def make_native_features(timeline, raw_q, gain, p_group=None, normalizer=None):
    p_valid_common = timeline['p1_valid'] & np.repeat(timeline['p10_valid'], 10)
    q_valid_common = raw_q['q1_valid'][gain] & np.repeat(raw_q['q10_valid'][gain], 10)
    if p_group is None:
        p_group = causal_signal_features(timeline['p1'], p_valid_common, timeline['continuity1'])
    q_group = causal_signal_features(raw_q['q1'][gain], q_valid_common, timeline['continuity1'])
    features = np.zeros((len(timeline['p1']), 18), np.float32)
    features[:, :6] = p_group
    features[:, 12:] = q_group
    assert not np.count_nonzero(features[:, 6:12]), 'All six S channels must remain zero.'
    if normalizer is not None:
        features = apply_normalizer_1s(features, normalizer)
    return (features, p_group, q_group)

## 3. Normalization and eligible windows

Training-period means and standard deviations apply only to values with valid masks. Missing values become zero; mask channels remain separate. The saved parameters are available with the model attachments. A new training run fits its own parameters on the training partition.

The 1 s model uses 640 s of history and produces one logit per 10 s endpoint. F0 has 42 channels, including 24 zero-valued optional morphology channels. Eligible endpoints require valid labels, native/grid observations and continuous history; prediction replay also requires label and segment metadata.


In [ ]:
def norm_apply(raw, n):
    x = raw.astype(np.float32, copy=True)
    mu = np.array(n['means'])
    sd = np.array(n['stds'])
    for g in (0, 6, 12):
        for j in range(3):
            v, m = (g + j, g + 3 + j)
            ok = x[:, m] > 0.5
            x[:, v] = 0
            x[ok, v] = ((raw[ok, v] - mu[v]) / sd[v]).astype(np.float32)
    return x

def endpoints_10s(t, left, right):
    e = np.arange(left + T - 1, right, dtype=np.int64)
    ok = t['valid'][e].astype(bool) & t['endpoint_valid'][e].astype(bool)
    ok &= t['continuity'][e] == t['continuity'][e - T + 1]
    return e[ok]

In [ ]:
def apply_normalizer_1s(features, normalizer):
    out = features.copy()
    for key, info in normalizer.items():
        column = int(key)
        good = out[:, column + 3].astype(bool)
        out[good, column] = (out[good, column] - info['mean']) / info['std']
        out[~good, column] = 0.0
    return out

def endpoint_indices(timeline, left, right):
    endpoint = np.arange(left + 63, right, dtype=np.int64)
    second = endpoint * 10 + 9
    valid = timeline['valid'][endpoint] & timeline['p10_valid'][endpoint] & timeline['p1_valid'][second] & (timeline['continuity10'][endpoint] == timeline['continuity10'][endpoint - 63]) & (timeline['continuity1'][second] == timeline['continuity1'][second - 639])
    return endpoint[valid]

In [ ]:
def full_window(features18, values12, masks12, endpoints, with_shape):
    seconds = np.asarray(endpoints, dtype=np.int64) * 10 + 9
    b = len(seconds)
    out = np.zeros((b, 42, 640), np.float32)
    for i, s in enumerate(seconds):
        out[i, :18] = features18[s - 639:s + 1].T
        if with_shape:
            out[i, 18:30] = values12[s - 639:s + 1].T
            out[i, 30:42] = masks12[s - 639:s + 1].T
    return out

def provider_h1(endpoints, shape):
    return full_window(BASE18, EXTRA_VALUES, EXTRA_MASKS, endpoints, shape)

In [ ]:
def trusted_artifact(case_id, relative_path):
    if not ARTIFACT_ROOT.strip():
        raise ValueError('Set ARTIFACT_ROOT to extracted verified attachments')
    base = Path(ARTIFACT_ROOT).expanduser().resolve()
    published = f'results/{case_id}/{relative_path}'
    path = (base / published).resolve()
    if not path.is_relative_to(base):
        raise ValueError('Artifact path leaves its root')
    source = json.loads((ROOT / 'results' / case_id / 'source.json').read_text(encoding='utf-8'))
    matches = [r for r in source['selected_files'] if r['published_path'] == published]
    if len(matches) != 1:
        raise ValueError('File has no unique trusted source hash')
    expected = matches[0]['published_sha256']
    if not path.is_file():
        raise FileNotFoundError(f'Missing selected attachment{published}')
    with path.open('rb') as handle:
        actual = hashlib.file_digest(handle, 'sha256').hexdigest()
    if actual != expected:
        raise ValueError('Selected attachment SHA-256 mismatch')
    return (path, expected)
config10 = json.loads((ROOT / 'results' / 'ukdale-10s' / 'config.json').read_text(encoding='utf-8'))
config1 = json.loads((ROOT / 'results' / 'ukdale-1s' / 'config.json').read_text(encoding='utf-8'))
print({'10s_probability_threshold': config10['threshold'], '1s_logit_thresholds': config1['thresholds'], 'all_seeds': list(SEEDS)})
if RUN_MODEL_LOAD:
    normal10_path, _ = trusted_artifact('ukdale-10s', 'normalizer.json')
    normal1_path, _ = trusted_artifact('ukdale-1s', 'normalizer.json')
    normal10 = json.loads(normal10_path.read_text(encoding='utf-8'))
    normal1 = json.loads(normal1_path.read_text(encoding='utf-8'))
    print({'10s_value_parameters': len(normal10['means']), '1s_normalized_columns': list(normal1)})
else:
    print('Normalization files: external model attachments.')

## 4. Causal TCN architectures

A temporal convolutional network learns patterns across a history window, such as a sustained power increase following a switching event. Dilated convolutions extend the time span covered by a filter; left padding prevents future samples from entering a prediction. The network returns a logit, an unbounded ON-state score used by the loss and thresholding functions.

V6 uses four residual blocks, width 32, dilations `(1,2,4,8)` and a receptive field of 61 bins. V11 uses width 22 and eight dilations `(1,2,4,8,16,32,32,64)` on 42 × 640 inputs. Both classify the final time step. Parameter names and forward computations match the corresponding saved checkpoints.


In [ ]:
if TORCH_AVAILABLE:

    class CausalConv1d(nn.Module):

        def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
            super().__init__()
            self.left_padding = (kernel_size - 1) * dilation
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=0)

        def forward(self, x):
            return self.conv(F.pad(x, (self.left_padding, 0)))

    class CausalResidualBlock(nn.Module):

        def __init__(self, in_channels, out_channels, dilation, dropout=0.1):
            super().__init__()
            self.conv1 = CausalConv1d(in_channels, out_channels, 3, dilation)
            self.conv2 = CausalConv1d(out_channels, out_channels, 3, dilation)
            self.dropout = nn.Dropout(dropout)
            self.residual = nn.Identity() if in_channels == out_channels else nn.Conv1d(in_channels, out_channels, 1)

        def forward(self, x):
            hidden = self.dropout(F.relu(self.conv1(x)))
            hidden = self.dropout(F.relu(self.conv2(hidden)))
            return F.relu(hidden + self.residual(x))

    class RichCausalTCN(nn.Module):

        def __init__(self):
            super().__init__()
            blocks = []
            channels = 18
            for dilation in (1, 2, 4, 8):
                blocks.append(CausalResidualBlock(channels, 32, dilation, 0.1))
                channels = 32
            self.blocks = nn.Sequential(*blocks)
            self.head = nn.Conv1d(32, 1, 1)

        def forward_sequence(self, x):
            return self.head(self.blocks(x))[:, 0, :]

        def seq(self, x):
            return self.forward_sequence(x)

        def forward(self, x):
            return self.forward_sequence(x)[:, -1]

        @property
        def receptive_field(self):
            return 61

        @property
        def rf(self):
            return self.receptive_field

    def model():
        return RichCausalTCN()

In [ ]:
if TORCH_AVAILABLE:

    class CausalConv(nn.Module):

        def __init__(self, a, b, k=3, d=1):
            super().__init__()
            self.pad = (k - 1) * d
            self.c = nn.Conv1d(a, b, k, dilation=d)

        def forward(self, x):
            return self.c(F.pad(x, (self.pad, 0)))

    class RB(nn.Module):

        def __init__(self, a, b, d):
            super().__init__()
            self.a = CausalConv(a, b, 3, d)
            self.b = CausalConv(b, b, 3, d)
            self.s = nn.Identity() if a == b else nn.Conv1d(a, b, 1)
            self.drop = nn.Dropout(0.1)

        def forward(self, x):
            return F.relu(self.s(x) + self.drop(F.relu(self.b(self.drop(F.relu(self.a(x)))))))

    class M0TCN(nn.Module):

        def __init__(self):
            super().__init__()
            ch = 42
            self.ds = (1, 2, 4, 8, 16, 32, 32, 64)
            self.blocks = nn.Sequential(*[RB(42 if i == 0 else 22, 22, d) for i, d in enumerate(self.ds)])
            self.out = nn.Conv1d(22, 1, 1)

        def forward_sequence(self, x):
            return self.out(self.blocks(x))[:, 0]

        def forward(self, x):
            return self.forward_sequence(x)[:, -1]

In [ ]:
if RUN_MODEL_LOAD:
    if not TORCH_AVAILABLE:
        raise ImportError('Install the documented PyTorch runtime')
    from wattsign.selected_models import load_model
    checkpoint, expected = trusted_artifact('ukdale-1s', f'models/M0_TCN_F0_seed{CHECKPOINT_SEED}.pt')
    loaded_model = load_model('ukdale-1s', checkpoint, expected_sha256=expected)
    print(type(loaded_model).__name__, 'loaded on CPU.')
else:
    print('Checkpoint loading: disabled (RUN_MODEL_LOAD=False).')

## 5. Loss, sampling and early stopping

Both protocols optimize binary cross-entropy with logits using AdamW (`lr=1e-3`, `weight_decay=1e-4`), batch size 64, at most 40 epochs and patience 5. Early stopping uses class-balanced BCE on the calibration partition. The chronological split is 60/20/20, with all five seeds evaluated separately.

V6 builds a balanced endpoint list and deterministic initial state for each seed; its epoch permutation uses `seed + 10000`. V11 uses `seed*100000+epoch` for its 40 precomputed batch orders. The [V6](../research/ukdale_v6.py) and [V11](../research/ukdale_v11.py) workflows prepare these inputs from mains and dedicated microwave records, then call the training functions below for each seed.

V6's early-stop scoring runs under `no_grad` while dropout remains in training mode. V11 explicitly uses evaluation mode and preserves RNG state. This distinction is part of the two protocols. The recorded experiments used CUDA; CPU training is supported, but numerical reruns depend on hardware, dependencies and input versions.


In [ ]:
VALUE_MASK_PAIRS = [(g + j, g + 3 + j) for g in (0, 6, 12) for j in range(3)]

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def array_sha256(array: np.ndarray) -> str:
    x = np.ascontiguousarray(array)
    return sha256_bytes(f'{x.dtype}|{x.shape}|'.encode() + x.view(np.uint8).tobytes())

def fit_masked_normalizer(features: np.ndarray, fit_end: int) -> dict:
    means, stds = (np.zeros(18, np.float64), np.ones(18, np.float64))
    for value_col, mask_col in VALUE_MASK_PAIRS:
        ok = features[:fit_end, mask_col] > 0.5
        values = features[:fit_end, value_col][ok].astype(np.float64)
        assert values.size > 100
        means[value_col], stds[value_col] = (values.mean(), max(values.std(), 1e-06))
    return {'means': means.tolist(), 'stds': stds.tolist(), 'fit_end': int(fit_end), 'fit_hash': array_sha256(features[:fit_end])}

def feature_normalizer(features, end_second):
    result = {}
    for column in (0, 1, 2, 12, 13, 14):
        good = features[:end_second, column + 3].astype(bool)
        values = features[:end_second, column][good]
        assert values.size
        result[str(column)] = {'mean': float(values.mean()), 'std': max(float(values.std()), 1e-06)}
    return result

def read_timeline_10s(spec):
    raw = scan_psq_10s(spec['mains'], spec['start'], spec['end'])
    scan = scan_microwave_native(spec['microwave'], spec['start'], spec['end'])
    labels = project_labels_to_10s(scan, spec['start'], spec['end'])
    last_second = np.arange(len(raw['p']), dtype=np.int64) * 10 + 9
    eligible = raw['p_valid'] & raw['p1_valid'][last_second]
    mature = last_second >= 639
    eligible[mature] &= raw['continuity1'][last_second[mature]] == raw['continuity1'][last_second[mature] - 639]
    features = pack_18_channels([(raw['p'], raw['p_valid']), (raw['s'], raw['s_valid']), (raw['q'], raw['q_valid'])], raw['continuity10'])
    return {'features_unscaled': features, 'continuity': raw['continuity10'], 'endpoint_valid': eligible.astype(np.uint8), 'p': raw['p'], 's': raw['s'], 'q': raw['q'], **labels}

def read_timeline_1s(spec):
    raw = scan_psq_seconds(spec['mains'], spec['start'], spec['end'], gains=(1.0,))
    labels = scan_microwave_labels(spec['microwave'], spec['start'], spec['end'])
    timeline = {k: raw[k] for k in ('p1', 'p1_valid', 'p10', 'p10_valid', 'continuity1', 'continuity10')}
    timeline.update({k: labels[k] for k in ('target', 'valid', 'stable', 'event')})
    timeline['start_s'] = spec['start']
    return (timeline, {k: raw[k] for k in ('q1', 'q10', 'q1_valid', 'q10_valid')})

def balanced_samples(endpoints, labels, seed):
    positive = endpoints[labels[endpoints] == 1]
    negative = endpoints[labels[endpoints] == 0]
    if len(positive) == 0 or len(negative) < len(positive):
        raise ValueError('Balanced sampling requires positive examples and enough negative examples')
    rng = np.random.default_rng(seed)
    chosen = rng.permutation(negative)[:len(positive)]
    sample = np.concatenate([positive, chosen])
    return sample[rng.permutation(len(sample))]

def ensure_two_classes(labels, endpoints, name):
    if len(endpoints) == 0 or set(np.unique(labels[endpoints]).tolist()) != {0, 1}:
        raise ValueError(f'{name} needs nonempty positive and negative eligible endpoints')
apply_normalizer = apply_normalizer_1s

In [ ]:
def sj(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'))

def sh(x):
    return hashlib.sha256(x).hexdigest()

def ah(x):
    x = np.ascontiguousarray(x)
    return sh(f'{x.dtype}|{x.shape}|'.encode() + x.view(np.uint8).tobytes())

def sth(d):
    h = hashlib.sha256()
    for k in sorted(d):
        x = d[k].detach().cpu().contiguous()
        h.update(k.encode())
        h.update(str(x.dtype).encode())
        h.update(str(tuple(x.shape)).encode())
        h.update(x.numpy().view(np.uint8).tobytes())
    return h.hexdigest()

def reset(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

In [ ]:
def mode(z, m):
    z = z.copy()
    if m == 'A':
        z[:, 6:] = 0
    elif m == 'PS':
        z[:, 12:] = 0
    elif m == 'PQ':
        z[:, 6:12] = 0
    elif m == 'M':
        z[:, 6:9] = 0
        z[:, 12:15] = 0
    elif m in ('B_snapshot', 'B\u5feb\u7167'):
        z[:, :, :63] = 0
    elif m == 'B':
        pass
    else:
        raise ValueError(m)
    return z

def edges(c, l, r):
    return np.r_[l, np.flatnonzero(c[l + 1:r] != c[l:r - 1]) + l + 1, r]

def bce(y, prob):
    log = np.log(np.clip(prob, 1e-07, 1 - 1e-07) / (1 - np.clip(prob, 1e-07, 1 - 1e-07)))
    l = np.logaddexp(0, log) - y * log
    return float(0.5 * l[y == 1].mean() + 0.5 * l[y == 0].mean())

In [ ]:
if TORCH_AVAILABLE:

    class DS(Dataset):

        def __init__(self, e, m, feat=None, t=None):
            self.e = np.asarray(e)
            self.m = m
            self.feat = globals()['x'] if feat is None else feat
            self.t = globals()['h1'] if t is None else t

        def __len__(self):
            return len(self.e)

        def __getitem__(self, i):
            e = self.e[i]
            q = mode(np.ascontiguousarray(self.feat[e - T + 1:e + 1].T[None]), self.m)[0]
            return (torch.from_numpy(q), torch.tensor(np.float32(self.t['target'][e])))

    @torch.no_grad()
    def seqinfer(m, feat, cont, e, l, r, kind):
        out = np.empty(len(e), np.float32)
        e = np.asarray(e)
        for a, b in zip(edges(cont, l, r)[:-1], edges(cont, l, r)[1:], strict=True):
            lo, hi = (np.searchsorted(e, a), np.searchsorted(e, b))
            if lo == hi:
                continue
            local = e[lo:hi]
            for cl in range(int(a), int(b), 65536):
                cr = min(int(b), cl + 65536)
                ilo, ihi = (np.searchsorted(local, cl), np.searchsorted(local, cr))
                if ilo == ihi:
                    continue
                h = max(int(a), cl - m.rf + 1)
                q = mode(np.ascontiguousarray(feat[h:cr].T[None]), kind)
                v = m.seq(torch.from_numpy(q).to(DEVICE))[0]
                ii = torch.as_tensor(local[ilo:ihi] - h, device=DEVICE)
                out[lo + ilo:lo + ihi] = v[ii].cpu().numpy()
        return torch.sigmoid(torch.from_numpy(out)).numpy().astype(np.float32)

    @torch.no_grad()
    def wininfer(m, feat, e, kind, donor=None):
        ans = []
        e = np.asarray(e)
        for i in range(0, len(e), IBATCH):
            ee = e[i:i + IBATCH]
            ix = ee[:, None] - np.arange(T - 1, -1, -1)
            q = np.ascontiguousarray(feat[ix].transpose(0, 2, 1))
            if donor is not None:
                dd = np.asarray(donor[i:i + len(ee)])
                dx = dd[:, None] - np.arange(T - 1, -1, -1)
                q[:, 6:, :] = feat[dx][:, :, 6:].transpose(0, 2, 1)
            ans.append(torch.sigmoid(m(torch.from_numpy(mode(q, kind)).to(DEVICE))).cpu().numpy())
        return np.concatenate(ans).astype(np.float32)

In [ ]:
if TORCH_AVAILABLE:
    def ckpath(n, s):
        return CP / f'{n}_seed{s}.pt'

    def train_v6_original(n, s):
        require_execution()
        reset(s)
        m = model().to(DEVICE)
        m.load_state_dict(ini[s])
        assert sth(m.state_dict()) == rep['initial_state_hashes'][str(s)]
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
        ds = DS(samples[s], n, feat=x, t=h1)
        rng = np.random.default_rng(s + 10000)
        best = np.inf
        bst = None
        be = 0
        stale = 0
        hist = []
        start = time.perf_counter()
        torch.cuda.reset_peak_memory_stats() if DEVICE.type == 'cuda' else None
        for ep in range(1, MAXE + 1):
            m.train()
            order = rng.permutation(len(ds))
            dl = DataLoader(Subset(ds, order.tolist()), batch_size=BATCH, shuffle=False, num_workers=0)
            ls = []
            for xb, yb in dl:
                xb, yb = (xb.to(DEVICE), yb.to(DEVICE))
                opt.zero_grad(set_to_none=True)
                loss = F.binary_cross_entropy_with_logits(m(xb), yb)
                loss.backward()
                opt.step()
                ls.append(float(loss.detach().cpu()))
            pr = wininfer(m, x, cal, n) if n in ('B_snapshot', 'B\u5feb\u7167') else seqinfer(m, x, h1['continuity'], cal, *CALIBRATION_BOUNDS, n)
            val = bce(caly, pr)
            hist.append({'model': n, 'seed': s, 'epoch': ep, 'training_bce': np.mean(ls), 'calibration_balanced_bce': val})
            if val < best - 1e-09:
                best = val
                bst = copy.deepcopy(m.state_dict())
                be = ep
                stale = 0
            else:
                stale += 1
                if stale >= PATIENCE:
                    break
        m.load_state_dict(bst)
        meta = {'model': n, 'seed': s, 'best_epoch': be, 'best_balanced_bce': best, 'training_seconds': time.perf_counter() - start, 'peak_memory_mb': torch.cuda.max_memory_allocated() / 1024 ** 2 if DEVICE.type == 'cuda' else 0.0, 'initial_state_hash': sth(ini[s]), 'state_hash': sth(bst), 'sampling_hash': ah(samples[s]), 'normalizer_hash': sh(sj(normal).encode())}
        torch.save({'state_dict': bst, 'metadata': meta}, ckpath(n, s))
        return (meta, pd.DataFrame(hist))

In [ ]:
def array_hash(x):
    x = np.ascontiguousarray(x)
    return hashlib.sha256((str(x.dtype) + str(x.shape)).encode() + x.view(np.uint8).tobytes()).hexdigest()

def state_hash(s):
    h = hashlib.sha256()
    for k in sorted(s):
        h.update(k.encode())
        h.update(s[k].detach().cpu().contiguous().numpy().view(np.uint8).tobytes())
    return h.hexdigest()

def reset_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def capture_rng():
    return {'cpu': torch.get_rng_state().clone(), 'cuda': [x.clone() for x in torch.cuda.get_rng_state_all()]}

def restore_rng(r):
    torch.set_rng_state(r['cpu'])
    torch.cuda.set_rng_state_all(r['cuda'])

def same_rng(a, b):
    return torch.equal(a['cpu'], b['cpu']) and all((torch.equal(x, y) for x, y in zip(a['cuda'], b['cuda'], strict=True)))

In [ ]:
def bce_balanced(y, z):
    l = np.logaddexp(0, z) - np.asarray(y) * z
    return float(0.5 * (l[np.asarray(y) == 1].mean() + l[np.asarray(y) == 0].mean()))

In [ ]:
if TORCH_AVAILABLE:

    def stream_slice(source, left, right, shape):
        base, values, masks, _ = source
        out = np.zeros((1, 42, right - left), np.float32)
        out[0, :18] = base[left:right].T
        if shape:
            out[0, 18:30] = values[left:right].T
            out[0, 30:42] = masks[left:right].T
        return torch.from_numpy(np.ascontiguousarray(out)).to(DEVICE)

    def infer(model, provider, endpoints, shape, source=None, bs=512):
        was = model.training
        rng = capture_rng()
        model.eval()
        endpoints = np.asarray(endpoints, np.int64)
        out = np.empty(len(endpoints), np.float32)
        try:
            with torch.inference_mode():
                if source is None:
                    for i in range(0, len(endpoints), bs):
                        out[i:i + bs] = model(torch.from_numpy(provider(endpoints[i:i + bs], shape)).to(DEVICE)).float().cpu().numpy()
                else:
                    _, _, _, continuity = source
                    seconds = endpoints * 10 + 9
                    cuts = np.r_[0, np.flatnonzero(continuity[seconds[1:]] != continuity[seconds[:-1]]) + 1, len(endpoints)]
                    for lo0, hi0 in zip(cuts[:-1], cuts[1:], strict=True):
                        for lo in range(lo0, hi0, bs):
                            hi = min(hi0, lo + bs)
                            sec = seconds[lo:hi]
                            left = int(sec[0] - 639)
                            right = int(sec[-1] + 1)
                            x = stream_slice(source, left, right, shape)
                            if isinstance(model, M0TCN):
                                logits = model.forward_sequence(x)[0, torch.as_tensor(sec - left, device=DEVICE)]
                            else:
                                raise ValueError('This selected workflow supports M0_TCN only')
                            out[lo:hi] = logits.float().cpu().numpy()
            return out
        finally:
            model.train(was)
            assert same_rng(rng, capture_rng())

In [ ]:
if TORCH_AVAILABLE:
    def train_v11_original(arch, shape, seed, repetition, phase):
        require_execution()
        reset_all(seed)
        model = MODELS[arch]().to(DEVICE)
        initial_hash = state_hash(model.state_dict())
        key = (arch, seed)
        assert key not in INITIAL_STATE_HASHES or INITIAL_STATE_HASHES[key] == initial_hash, f'Initial state differs: {key}'
        INITIAL_STATE_HASHES[key] = initial_hash
        opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
        best = np.inf
        state = None
        wait = 5
        hist = []
        start = time.perf_counter()
        torch.cuda.reset_peak_memory_stats() if DEVICE.type == 'cuda' else None
        for epoch in range(40):
            model.train()
            losses = []
            order = BATCH_ORDERS[seed][epoch]
            for st in range(0, len(order), 64):
                ep = SAMPLES[seed][order[st:st + 64]]
                x = torch.from_numpy(provider_h1(ep, shape)).to(DEVICE)
                yy = torch.from_numpy(h1['target'][ep].astype(np.float32)).to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = F.binary_cross_entropy_with_logits(model(x), yy)
                loss.backward()
                opt.step()
                losses.append(float(loss.detach().cpu()))
            z = infer(model, provider_h1, PARTITIONS['calibration'], shape, H1_SOURCE)
            val = bce_balanced(h1['target'][PARTITIONS['calibration']], z)
            hist.append({'epoch': epoch + 1, 'training_loss': np.mean(losses), 'calibration_balanced_bce': val})
            if val < best - 1e-08:
                best = val
                state = copy.deepcopy(model.state_dict())
                be = epoch + 1
                wait = 5
            else:
                wait -= 1
                if not wait:
                    break
        model.load_state_dict(state)
        meta = {'architecture': arch, 'representation': 'F1' if shape else 'F0', 'seed': seed, 'repetition': repetition, 'initial_state_hash': initial_hash, 'best_epoch': be, 'training_seconds': time.perf_counter() - start, 'peak_memory_mb': torch.cuda.max_memory_allocated() / 2 ** 20 if DEVICE.type == 'cuda' else 0.0, 'state_hash': state_hash(state), 'parameter_count': sum((p.numel() for p in model.parameters()))}
        torch.save({'state': state, 'metadata': meta}, OUT / 'checkpoints' / f'{phase}_cold{repetition}_{arch}_F{int(shape)}_seed{seed}.pt')
        return (model, meta, hist)

### Prepare the training context

Preparation reads and labels the raw records, applies the chronological split, fits normalization on training data, constructs balanced samples and fixes initialization and batch orders. Each workflow uses its own runtime namespace.


In [ ]:
def prepare_training_10s(spec):
    global h1, x, normal, samples, ini, rep, cal, caly, CALIBRATION_BOUNDS
    h1 = read_timeline_10s(spec)
    count = len(h1['target'])
    training_end = int(0.6 * count)
    calibration_end = int(0.8 * count)
    splits = {'train': (0, training_end), 'calibration': (training_end, calibration_end), 'h1_final': (calibration_end, count)}
    partitions = {name: endpoints_10s(h1, *bounds) for name, bounds in splits.items()}
    for name, e in partitions.items():
        ensure_two_classes(h1['target'], e, name)
    normal = fit_masked_normalizer(h1['features_unscaled'], training_end)
    x = norm_apply(h1['features_unscaled'], normal)
    samples = {s: balanced_samples(partitions['train'], h1['target'], s) for s in SEEDS}
    ini = {}
    for s in SEEDS:
        reset(s)
        ini[s] = copy.deepcopy(model().state_dict())
    rep = {'initial_state_hashes': {str(s): sth(ini[s]) for s in SEEDS}}
    cal = partitions['calibration']
    caly = h1['target'][cal].astype(np.uint8)
    CALIBRATION_BOUNDS = splits['calibration']
    return (partitions, splits)

def prepare_training_1s(spec):
    global h1, BASE18, EXTRA_VALUES, EXTRA_MASKS, NORMALIZER, SAMPLES, BATCH_ORDERS, PARTITIONS, H1_SOURCE
    h1, raw_q = read_timeline_1s(spec)
    count = len(h1['target'])
    train_end = int(0.6 * count)
    cal_end = int(0.8 * count)
    splits = {'train': (0, train_end), 'calibration': (train_end, cal_end), 'h1_final': (cal_end, count)}
    PARTITIONS = {name: endpoint_indices(h1, *bounds) for name, bounds in splits.items()}
    for name, e in PARTITIONS.items():
        ensure_two_classes(h1['target'], e, name)
    unnormalized, _, _ = make_native_features(h1, raw_q, 1.0)
    NORMALIZER = feature_normalizer(unnormalized, train_end * 10)
    BASE18 = apply_normalizer(unnormalized, NORMALIZER)
    EXTRA_VALUES = np.broadcast_to(np.zeros((1, 12), np.float32), (len(BASE18), 12))
    EXTRA_MASKS = np.broadcast_to(np.zeros((1, 12), np.uint8), (len(BASE18), 12))
    H1_SOURCE = (BASE18, EXTRA_VALUES, EXTRA_MASKS, h1['continuity1'])
    SAMPLES = {s: balanced_samples(PARTITIONS['train'], h1['target'], s) for s in SEEDS}
    BATCH_ORDERS = {s: [np.random.default_rng(s * 100000 + epoch).permutation(len(SAMPLES[s])).astype(np.int64) for epoch in range(40)] for s in SEEDS}
    return (splits, raw_q)

### Train, calibrate and evaluate

Set `DATASET_ROOT`, `MICROWAVE_CHANNEL_FILE`, a new `NEW_OUTPUT_ROOT` and the required execution flags. The selected workflow trains all five seeds and saves normalization, sampling/order records, checkpoints, training history, calibration thresholds, fixed-threshold test metrics and predictions. Outputs are written separately from the recorded experiments.

`EVALUATION_SPECS` accepts additional partitions as dictionaries with `name`, `mains`, `microwave`, `start` and `end`, such as H1-later or H2. Evaluation uses the training normalizer and H1-calibration thresholds. The default training interval is the H1 90-day period. Native 1 s preparation can require several GiB of RAM; CPU training can be slow. Dependency versions are listed in [runtime](../docs/en/runtime.md).


In [ ]:
TRAIN_START_UTC_S = 1363564800
TRAIN_END_UTC_S = 1371340800

if RUN_TRAINING:
    if not RUN_DATA_READ:
        raise RuntimeError('Complete training reads raw data; set RUN_DATA_READ=True explicitly')
    if not DATASET_ROOT.strip() or not MICROWAVE_CHANNEL_FILE.strip() or not NEW_OUTPUT_ROOT.strip():
        raise ValueError('Set DATASET_ROOT, MICROWAVE_CHANNEL_FILE and a new NEW_OUTPUT_ROOT')
    dataset = Path(DATASET_ROOT).expanduser().resolve()
    microwave = Path(MICROWAVE_CHANNEL_FILE).expanduser()
    if not microwave.is_absolute():
        microwave = dataset / 'house_1' / microwave
    spec = {'mains':dataset / 'house_1' / 'mains.dat', 'microwave':microwave,
            'start':TRAIN_START_UTC_S, 'end':TRAIN_END_UTC_S}
    if str(ROOT / 'research') not in sys.path:
        sys.path.insert(0, str(ROOT / 'research'))
    if CASE_TO_TRAIN == 'ukdale-10s':
        import ukdale_v6 as selected_recipe
    elif CASE_TO_TRAIN == 'ukdale-1s':
        import ukdale_v11 as selected_recipe
    else:
        raise ValueError('Select ukdale-10s or ukdale-1s')
    protected = [dataset]
    if ARTIFACT_ROOT.strip():
        protected.append(Path(ARTIFACT_ROOT).expanduser().resolve())
    new_run = selected_recipe.run(spec, NEW_OUTPUT_ROOT,
        evaluation_specs=EVALUATION_SPECS, protected_roots=protected,
        device=TRAINING_DEVICE, execute=True,
        trainer=train_v6_original if CASE_TO_TRAIN == 'ukdale-10s' else train_v11_original)
    print({'new_output':new_run['output'], 'trained_seeds':new_run['seeds']})
else:
    print('Training: disabled (RUN_TRAINING=False).')

## 6. Thresholds and evaluation

Calibration selects the highest threshold meeting 95% recall. V6 stores a probability threshold; V11 stores a logit threshold for each seed. These score scales require their corresponding thresholds. The calibration functions are called by the training workflow, not by the default result display.

AP uses continuous scores; Recall, F1 and FPR use fixed-threshold decisions. High-power OFF FPR is conditioned on a negative microwave label and aggregate P ≥ 800 W. Conditional concordance compares ON and OFF scores in the same 50 W power bin, then averages by event; unmatched events affect coverage.


In [ ]:
def pooled_recall_10s(y, s, t):
    yy = np.tile(y, s.shape[0])
    ss = s.ravel()
    return float((ss[yy == 1] >= t).mean())

def historical_threshold_10s(y, s, target_recall=0.95):
    """Select a pooled calibration threshold for the requested recall."""
    yy = np.tile(y, s.shape[0])
    ss = s.ravel()
    pos = np.sort(ss[yy == 1])
    can = np.r_[np.nextafter(ss.min(), -np.inf), np.unique(ss), np.nextafter(ss.max(), np.inf)]
    assert can[0] < ss.min() and can[-1] > ss.max()
    need = min(max(math.ceil(target_recall * len(pos) - 1e-12), 1), len(pos))
    t = float(pos[-need])
    assert pooled_recall_10s(y, s, t) >= target_recall
    return t

def binary_metrics_10s(y, s, t):
    q = (s >= t).astype(np.uint8)
    a, b, c, _ = precision_recall_fscore_support(y, q, average='binary', zero_division=0)
    z = confusion_matrix(y, q, labels=[0, 1])
    return {'PR-AUC': float(average_precision_score(y, s)), 'ROC-AUC': float(roc_auc_score(y, s)), 'Precision': float(a), 'Recall': float(b), 'F1': float(c), 'balanced_accuracy': float(balanced_accuracy_score(y, q)), 'TN': int(z[0, 0]), 'FP': int(z[0, 1]), 'FN': int(z[1, 0]), 'TP': int(z[1, 1])}

def off_fpr_10s(y, s, p, t):
    r = {}
    for lab, lo, hi in [('<800 W', -np.inf, 800.0), ('800–1500 W', 800.0, 1500.0), ('1500–2500 W', 1500.0, 2500.0), ('≥2500 W', 2500.0, np.inf)]:
        k = (y == 0) & (p >= lo) & (p < hi)
        r['OFF_FPR ' + lab] = float((s[k] >= t).mean()) if k.any() else np.nan
    k = (y == 0) & (p >= 800)
    r['high_power_OFF_FPR'] = float((s[k] >= t).mean()) if k.any() else np.nan
    return r

In [ ]:
def historical_threshold_1s(y, s):
    p = np.sort(s[y == 1])
    need = math.ceil(0.95 * len(p))
    c = np.unique(np.r_[np.nextafter(s.min(), -np.inf), s, np.nextafter(s.max(), np.inf)])
    return float(c[len(p) - np.searchsorted(p, c, side='left') >= need][-1])

def condition(y, p, event, s):
    yy = np.asarray(y)
    pp = np.asarray(p)
    ee = np.asarray(event)
    ss = np.asarray(s)
    pos = np.flatnonzero((yy == 1) & (ee >= 0))
    all_events = np.unique(ee[pos])
    if not len(pos):
        return ({}, 0.0)
    off = np.flatnonzero(yy == 0)
    bins = np.floor(pp / 50.0).astype(np.int64)
    value_parts = []
    event_parts = []
    for b in np.unique(bins[pos]):
        refs = off[bins[off] == b]
        if not len(refs):
            continue
        ix = pos[bins[pos] == b]
        ranked = np.sort(ss[refs])
        left = np.searchsorted(ranked, ss[ix], side='left')
        right = np.searchsorted(ranked, ss[ix], side='right')
        value_parts.append((left + right).astype(np.float64) / (2.0 * len(ranked)))
        event_parts.append(ee[ix])
    if not value_parts:
        return ({}, 0.0)
    ev = np.concatenate(event_parts)
    value = np.concatenate(value_parts)
    ids, inv = np.unique(ev, return_inverse=True)
    total = np.bincount(inv, weights=value, minlength=len(ids))
    count = np.bincount(inv, minlength=len(ids))
    result = {int(e): float(a / b) for e, a, b in zip(ids, total, count, strict=True)}
    return (result, len(result) / max(1, len(all_events)))

def metric_table(y, p, q, event, s, thr):
    pred = s >= thr
    pr, re, f, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
    off = y == 0
    high = off & (p >= 800)
    cm = confusion_matrix(y, pred, labels=[0, 1])
    cond, cover = condition(y, p, event, s)
    result = {'AP': float(average_precision_score(y, s)), 'ROC-AUC': float(roc_auc_score(y, s)), 'Precision': float(pr), 'Recall': float(re), 'F1': float(f), 'balanced_accuracy': float(balanced_accuracy_score(y, pred)), 'high_power_OFF_FPR': float(pred[high].mean()) if high.any() else np.nan, 'power_conditioned_concordance': float(np.mean(list(cond.values()))) if cond else np.nan, 'matched_event_coverage': cover, 'OFF-P Spearman': float(spearmanr(s[off], p[off]).statistic), 'OFF-Q Spearman': float(spearmanr(s[off], q[off]).statistic), 'TN': int(cm[0, 0]), 'FP': int(cm[0, 1]), 'FN': int(cm[1, 0]), 'TP': int(cm[1, 1]), '_cond': cond}
    for name, left, right in [('＜800 W', -np.inf, 800), ('800–1500 W', 800, 1500), ('1500–2500 W', 1500, 2500), ('≥2500 W', 2500, np.inf)]:
        band = off & (p >= left) & (p < right)
        result[f'OFF_FPR {name}'] = float(pred[band].mean()) if band.any() else np.nan
    return result

## 7. Results and scope

**H1** and **H2** identify two recorded households; “H1 earlier” and “H1 later” identify separate time partitions. Each summary includes all five seeds rather than one selected initialization.

The 10-second TCN/PQ has high recall on H1, but its lower AP and F1 show that detecting most ON states can still come with many false detections. The 1-second TCN reaches AP **0.9661** and Recall **94.27%** on H2, with **0.656% high-power OFF FPR**; its lower scores on the H1 partitions show that performance depends on the recording and evaluation period.

H2 was exposed during development, so these values describe an exploratory household comparison. They do not establish performance in a previously unseen home. The two workflows have different preparation and evaluation protocols; their scores do not isolate the effect of changing input resolution.

V11's 1 s input resolution, 10 s output step and 640 s history refer to three different aspects of the workflow. Independent V6 pointwise prediction arrays were not saved, so that case supports aggregate-result inspection but not pointwise prediction replay.


In [ ]:
show_experiment('ukdale-10s')
show_experiment('ukdale-1s')

## Recorded metrics

**Average precision (AP)** summarizes how well ON states rank above OFF states across score thresholds. **Recall** is the share of valid ON endpoints detected at the fixed threshold; **F1** combines precision and recall. OFF false-positive rate (**OFF FPR**) counts the share of valid OFF endpoints predicted ON. The high-power variant checks false alarms when other household loads can resemble the microwave's power draw.

Values below are rounded from the case summaries and include all five protocol seeds. The results guide defines the exact metric denominators, aggregation, confidence intervals and limitations.

### ukdale-10s

[Source summary](../results/ukdale-10s/summary.json) · [English results](../docs/en/results.md)

| Partition | Model | Backend | AP | Recall | F1 |
|---|---|---|---:|---:|---:|
| H1 | TCN/PQ | — | 0.5573 | 0.9769 | 0.4650 |
| H2 | TCN/PQ | — | 0.8932 | 0.9597 | 0.8842 |

### ukdale-1s

[Source summary](../results/ukdale-1s/summary.json) · [English results](../docs/en/results.md)

| Partition | Model | Backend | AP | Recall | F1 |
|---|---|---|---:|---:|---:|
| H1 earlier | M0_TCN-F0 | — | 0.7975 | 0.9627 | 0.7394 |
| H1 later | M0_TCN-F0 | — | 0.6954 | 0.8953 | 0.7098 |
| H2 | M0_TCN-F0 | — | 0.9661 | 0.9427 | 0.9387 |


## Model files and predictions

[Runtime and artifacts](../docs/en/runtime.md)

For access to model files and saved predictions, contact **ruizhang.eric@gmail.com** with the case name, requested files and intended use. Raw datasets are obtained separately from their providers.
